In [1]:
from typing import TypedDict, Annotated, Literal, Optional
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import json
import re
import os

In [2]:
if "SILICONFLOW_API_KEY_LLM" not in os.environ:
    os.environ["SILICONFLOW_API_KEY_LLM"] = input("请输入LLM对应的硅基流动 API Key: ")

if "SILICONFLOW_API_KEY_LLM_JSON" not in os.environ:
    os.environ["SILICONFLOW_API_KEY_LLM_JSON"] = input("请输入LLM_JSON对应的硅基流动 API Key: ")

if "SILICONFLOW_API_KEY_EMB" not in os.environ:
    os.environ["SILICONFLOW_API_KEY_EMB"] = input("请输入embeddings模型硅基流动 API Key: ")

In [3]:

# ========== 1. 状态定义 ==========

class RAGDialogState(TypedDict):
    messages: Annotated[list[AnyMessage], "add_messages"]
    intent: Optional[str]
    slots: dict
    missing_slots: list[str]
    search_query: Optional[str]
    retrieved_docs: list[dict]
    final_answer: Optional[str]
    is_complete: bool


In [4]:

# ========== 2. 初始化组件 ==========

llm = ChatOpenAI(model="tencent/Hunyuan-MT-7B", 
                 temperature=0.7,
                 base_url="https://api.siliconflow.cn/v1",
                 api_key=os.getenv('SILICONFLOW_API_KEY_LLM'),)
llm_json = ChatOpenAI(model="tencent/Hunyuan-MT-7B", 
                      temperature=0.1,
                      base_url="https://api.siliconflow.cn/v1",
                      api_key=os.getenv("SILICONFLOW_API_KEY_LLM_JSON"),)



In [5]:

# ========== 银行知识库（Mock） ==========

def create_mock_vectorstore():
    texts = [
        "2024年全行净利润为120亿元，同比增长8%。",
        "2024年全行营业收入为680亿元，同比增长5%。",
        "2023年全行净利润为111亿元。",
        "2024年公司业务收入占比45%，零售业务收入占比55%。",
        "截至2024年底，不良贷款率为1.28%，较去年下降0.05个百分点。",
        "资本充足率为13.6%，核心一级资本充足率为10.2%。"
    ]
    # embeddings = OpenAIEmbeddings()
    embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",  # 或硅基流动支持的其他嵌入模型
    api_key=os.getenv("SILICONFLOW_API_KEY_EMB"),
    base_url="https://api.siliconflow.cn/v1",
    )
    return FAISS.from_texts(texts, embeddings)

vectorstore = create_mock_vectorstore()


In [6]:

# ========== 意图 - 槽位映射 ==========

INTENT_SLOT_MAP = {
    "查询业绩指标": ["时间", "指标", "机构"],
    "对比指标": ["时间", "对比时间", "指标", "机构"],
    "查询风险指标": ["时间", "指标"],
}


In [7]:

# ========== 3. 节点实现 ==========

def intent_classifier(state: RAGDialogState) -> RAGDialogState:
    messages = state["messages"]
    last_message = messages[-1].content if messages else ""

    prompt = ChatPromptTemplate.from_messages([
        ("system", """你是银行问数场景的对话理解助手，请从用户输入中提取：

1. 意图（intent）：[查询业绩指标, 对比指标, 查询风险指标, 其他]
2. 槽位（slots）：时间、对比时间、指标、机构
3. 缺失槽位（missing_slots）

输出严格 JSON：
{{
  "intent": "查询业绩指标",
  "slots": {{"时间": "今年", "指标": "净利润", "机构": "全行"}},
  "missing_slots": []
}}"""),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ])

    chain = prompt | llm_json
    response = chain.invoke({
        "history": messages[:-1],
        "input": last_message
    })

    try:
        content = response.content
        json_str = re.search(r'```json\s*(.*?)\s*```', content, re.DOTALL)
        if json_str:
            content = json_str.group(1)
        result = json.loads(content)
    except Exception:
        result = {"intent": "其他", "slots": {}, "missing_slots": []}

    merged_slots = {**state.get("slots", {}), **result.get("slots", {})}
    intent = result.get("intent", "其他")
    required = INTENT_SLOT_MAP.get(intent, [])
    missing = [s for s in required if s not in merged_slots]

    return {
        **state,
        "intent": intent,
        "slots": merged_slots,
        "missing_slots": missing
    }

def should_ask_followup(state: RAGDialogState) -> Literal["ask", "retrieve"]:
    return "ask" if state["missing_slots"] else "retrieve"

def generate_followup(state: RAGDialogState) -> RAGDialogState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """你是银行数据助手，请针对缺失信息生成追问：

意图：{intent}
已有信息：{slots}
缺失信息：{missing}

要求：一次只问一个关键问题，语气专业自然。"""),
        ("human", "生成追问")
    ])

    chain = prompt | llm
    response = chain.invoke({
        "intent": state["intent"],
        "slots": json.dumps(state["slots"], ensure_ascii=False),
        "missing": ", ".join(state["missing_slots"])
    })

    return {
        **state,
        "messages": state["messages"] + [AIMessage(content=response.content)],
        "final_answer": response.content
    }

def consolidate_and_search(state: RAGDialogState) -> RAGDialogState:
    parts = [state["intent"]]
    for k, v in state["slots"].items():
        parts.append(f"{k}:{v}")
    search_query = " ".join(parts)

    docs = vectorstore.similarity_search_with_score(search_query, k=3)
    retrieved = [
        {"content": d.page_content, "score": float(s)}  # State不能存numpy格式，所以需要转换
        for d, s in docs if s < 0.7
    ]

    return {
        **state,
        "search_query": search_query,
        "retrieved_docs": retrieved
    }

def check_retrieval_quality(state: RAGDialogState) -> Literal["generate", "fallback"]:
    return "generate" if state["retrieved_docs"] else "fallback"

def generate_fallback(state: RAGDialogState) -> RAGDialogState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """你是银行问数助手，当前未检索到匹配数据。
请引导用户调整查询条件（如时间、指标口径）。"""),
        ("human", "生成兜底回复")
    ])

    response = (prompt | llm).invoke({})

    return {
        **state,
        "messages": state["messages"] + [AIMessage(content=response.content)],
        "final_answer": response.content,
        "is_complete": True
    }

def generate_answer(state: RAGDialogState) -> RAGDialogState:
    context = "\n".join(
        f"- {d['content']}" for d in state["retrieved_docs"]
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", """你是银行业绩分析助手，仅基于给定资料回答。

用户需求：{slots}
可用资料：
{context}

要求：
1. 不得编造
2. 明确数值与时间
3. 语言专业简洁"""),
        ("human", "{input}")
    ])

    response = (prompt | llm).invoke({
        "slots": json.dumps(state["slots"], ensure_ascii=False),
        "context": context,
        "input": state["messages"][-1].content
    })

    return {
        **state,
        "messages": state["messages"] + [AIMessage(content=response.content)],
        "final_answer": response.content,
        "is_complete": True
    }


In [8]:

# ========== 4. 构建图 ==========

workflow = StateGraph(RAGDialogState)

workflow.add_node("intent_classifier_node", intent_classifier)
workflow.add_node("followup", generate_followup)
workflow.add_node("retrieve", consolidate_and_search)
workflow.add_node("generate", generate_answer)
workflow.add_node("fallback", generate_fallback)

workflow.add_edge(START, "intent_classifier_node")

workflow.add_conditional_edges(
    "intent_classifier_node",
    should_ask_followup,
    {"ask": "followup", "retrieve": "retrieve"}
)

workflow.add_edge("followup", END)

workflow.add_conditional_edges(
    "retrieve",
    check_retrieval_quality,
    {"generate": "generate", "fallback": "fallback"}
)

workflow.add_edge("generate", END)
workflow.add_edge("fallback", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
# app = workflow.compile()


In [9]:

# ========== 5. 使用示例 ==========

def chat_with_bot(user_input: str, thread_id: str = "default"):
    return app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config={"configurable": {"thread_id": thread_id}}
    )["final_answer"]

if __name__ == "__main__":
    print("【第1轮】今年全行净利润怎么样？")
    print(chat_with_bot("今年全行净利润怎么样？", "bank_001"))

    print("\n【第2轮】和去年比呢？")
    print(chat_with_bot("和去年比呢？", "bank_001"))


【第1轮】今年全行净利润怎么样？
根据提供的数据，2023年全行的净利润为111亿元，呈现出了一定的增长趋势。

【第2轮】和去年比呢？
请问您需要对比的是哪两个不同时间的净利润数据呢？


In [10]:
def chat_with_bot(user_input: str, thread_id: str = "default", verbose: bool = False):
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config={"configurable": {"thread_id": thread_id}}
    )
    
    if verbose:
        print("=" * 50)
        print(f"【完整状态】thread_id: {thread_id}")
        print(f"意图: {result.get('intent')}")
        print(f"槽位: {result.get('slots')}")
        print(f"缺失槽位: {result.get('missing_slots')}")
        print(f"检索查询: {result.get('search_query')}")
        print(f"检索文档数: {len(result.get('retrieved_docs', []))}")
        for i, doc in enumerate(result.get('retrieved_docs', []), 1):
            print(f"  [{i}] (score: {doc.get('score', 0):.3f}) {doc.get('content', '')[:50]}...")
        print(f"历史消息数: {len(result.get('messages', []))}")
        print("=" * 50)
    
    return result["final_answer"]

# 使用
if __name__ == "__main__":
    print("【第1轮】今年全行净利润怎么样？")
    print(chat_with_bot("今年全行净利润怎么样？", "bank_001", verbose=True))

    print("\n【第2轮】和去年比呢？")
    print(chat_with_bot("和去年比呢？", "bank_001", verbose=True))

    print("\n【第3轮】2023年")
    print(chat_with_bot("2023年", "bank_001", verbose=True))

【第1轮】今年全行净利润怎么样？
【完整状态】thread_id: bank_001
意图: 查询业绩指标
槽位: {'时间': '今年', '指标': '净利润', '机构': '全行'}
缺失槽位: []
检索查询: 查询业绩指标 时间:今年 指标:净利润 机构:全行
检索文档数: 1
  [1] (score: 0.485) 2023年全行净利润为111亿元。...
历史消息数: 2
根据提供的资料，2023年全行的净利润为111亿元。

【第2轮】和去年比呢？
【完整状态】thread_id: bank_001
意图: 对比指标
槽位: {'时间': '去年', '指标': '净利润', '机构': '全行'}
缺失槽位: ['对比时间']
检索查询: 查询业绩指标 时间:今年 指标:净利润 机构:全行
检索文档数: 1
  [1] (score: 0.485) 2023年全行净利润为111亿元。...
历史消息数: 2
请问您需要对比哪个指标的哪两个时间段的数据呢？

【第3轮】2023年
【完整状态】thread_id: bank_001
意图: 查询业绩指标
槽位: {'时间': '2023年', '指标': '净利润', '机构': '全行'}
缺失槽位: []
检索查询: 查询业绩指标 时间:2023年 指标:净利润 机构:全行
检索文档数: 2
  [1] (score: 0.491) 2023年全行净利润为111亿元。...
  [2] (score: 0.699) 截至2024年底，不良贷款率为1.28%，较去年下降0.05个百分点。...
历史消息数: 2
全行在2023年的净利润为111亿元。
